# Rennaigan backend on Colab

Runs the five detector services and the gateway on a Colab GPU and gives you a public
`https://….trycloudflare.com` address to paste into the UI (**Settings → ML Gateway → Gateway URL**).

Before running: **Runtime → Change runtime type → T4 GPU**. Then **Runtime → Run all**.
The first run takes about 8 minutes (model downloads). Keep this tab open while you use the UI.

Cell 3 prints the address. If something fails, run the last cell and copy its output.

After a code update: re-run cell 3. It pulls the latest code, restarts the backend and prints the address again (it stays the same while this session lives).

In [ ]:
#@title 1. Get the code
import os
if not os.path.exists("/content/Rennaigan"):
    !git clone -q --depth 1 -b backend-gateway https://github.com/Laxitha/Rennaigan.git /content/Rennaigan
os.chdir("/content/Rennaigan")
!git pull -q
!git log --oneline -1
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "No GPU: analysis will run in the slower reduced mode. For full speed: Runtime > Change runtime type > T4 GPU."

In [ ]:
#@title 2. One-time setup: tools, model repositories, packages and weights (about 6 minutes)
import os
os.chdir("/content/Rennaigan")
!bash scripts/colab_setup.sh

In [ ]:
#@title 3. Start the backend and get the public address
import os
os.chdir("/content/Rennaigan")

# Optional: a reasoned real/deepfake verdict written by Claude from the detector findings.
# Add your key under the key icon in Colab's left sidebar as a secret named ANTHROPIC_API_KEY
# (it starts with "sk-ant-") and switch on "Notebook access". Never paste the key into a cell.
try:
    from google.colab import userdata
    key = userdata.get("ANTHROPIC_API_KEY")
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key.strip()
        # Also send the image (or four video frames) to Claude for a visual review.
        # This shares uploaded media with Anthropic; delete the next line to send detector findings only.
        os.environ["RENNAIGAN_ASSESS_SEND_MEDIA"] = "1"
except Exception:
    pass  # no secret set: the rule-based verdict is used

!git pull -q
!python scripts/colab.py up

In [ ]:
#@title 4. Optional self-test: every detector on a real talking-head clip
import os
os.chdir("/content/Rennaigan")
!python scripts/colab.py selftest

In [ ]:
#@title Status and logs (run when something looks wrong)
import os
os.chdir("/content/Rennaigan")
!python scripts/colab.py status
!tail -15 /content/backend.log